In [13]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
import cv2
from pathlib import Path
from sklearn.metrics import cohen_kappa_score

carpeta_labeled = Path("../data/labeled")
carpeta_videos = Path("../data/raw")   # ajustado: los videos están aquí directo

UMBRAL_CIERRE = 0.5
UMBRAL_BOSTEZO = 3.0

In [14]:
xmls_a = sorted(carpeta_labeled.glob("*_A.xml"))
nombres_base = [p.stem.replace("_A", "") for p in xmls_a]

print(f"Se encontraron {len(nombres_base)} videos con par A/B:")
for n in nombres_base:
    existe_b = (carpeta_labeled / f"{n}_B.xml").exists()
    print(f"  {n} {'✓' if existe_b else '✗ FALTA EL B'}")

Se encontraron 18 videos con par A/B:
  V1_video1_normal ✓
  V1_video2_pocaluz ✓
  V2_video1_normal ✓
  V2_video2_lentes ✓
  V3_video1_normal ✓
  V3_video2_lentes ✓
  V4_video1_normal ✓
  V4_video2_pocaluz ✓
  V5_video1_normal ✓
  V5_video2_pocaluz ✓
  V6_video1_normal ✓
  V6_video2_pocaluz ✓
  V7_video1_normal ✓
  V7_video2_pocaluz ✓
  V8_video1_normal ✓
  V8_video2_lentes ✓
  V9_video1_normal ✓
  V9_video2_pocaluz ✓


In [15]:
def obtener_fps_frames(nombre_video):
    candidatos = list(carpeta_videos.glob(f"{nombre_video}.*"))
    if not candidatos:
        raise FileNotFoundError(f"No se encontró el video para {nombre_video}")
    cap = cv2.VideoCapture(str(candidatos[0]))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return fps, total

def xml_a_eventos(ruta_xml, fps):
    tree = ET.parse(ruta_xml)
    eventos = []
    for track in tree.getroot().findall(".//track"):
        label = track.get("label")
        boxes = track.findall("box")
        inicio = int(boxes[0].get("frame"))
        fin = int(boxes[-1].get("frame"))
        duracion = (fin - inicio) / fps
        eventos.append({"etiqueta": label, "frame_inicio": inicio, "frame_fin": fin, "duracion_seg": round(duracion, 2)})
    return eventos

def xml_a_vector(ruta_xml, etiqueta, total_frames):
    tree = ET.parse(ruta_xml)
    vector = np.zeros(total_frames, dtype=int)
    for track in tree.getroot().findall(".//track"):
        if track.get("label") != etiqueta:
            continue
        boxes = track.findall("box")
        inicio = int(boxes[0].get("frame"))
        fin = int(boxes[-1].get("frame"))
        vector[inicio:fin+1] = 1
    return vector

def validar_evento(etiqueta, duracion):
    if etiqueta == "cierre_ocular":
        return duracion >= UMBRAL_CIERRE
    elif etiqueta == "bostezo":
        return duracion >= UMBRAL_BOSTEZO
    return False

In [26]:
tabla_eventos_global = []
vectores_cierre_a, vectores_cierre_b = [], []
vectores_bostezo_a, vectores_bostezo_b = [], []
resumen_por_video = []

for nombre in nombres_base:
    ruta_a = carpeta_labeled / f"{nombre}_A.xml"
    ruta_b = carpeta_labeled / f"{nombre}_B.xml"

    fps, total_frames = obtener_fps_frames(nombre)

    eventos_a = xml_a_eventos(ruta_a, fps)
    for e in eventos_a:
        e["video"] = nombre
        e["etiquetador"] = "A"
        e["valido"] = validar_evento(e["etiqueta"], e["duracion_seg"])
        tabla_eventos_global.append(e)

    eventos_b = xml_a_eventos(ruta_b, fps)
    for e in eventos_b:
        e["video"] = nombre
        e["etiquetador"] = "B"
        e["valido"] = validar_evento(e["etiqueta"], e["duracion_seg"])
        tabla_eventos_global.append(e)

    v_a_c = xml_a_vector(ruta_a, "cierre_ocular", total_frames)
    v_b_c = xml_a_vector(ruta_b, "cierre_ocular", total_frames)
    v_a_y = xml_a_vector(ruta_a, "bostezo", total_frames)
    v_b_y = xml_a_vector(ruta_b, "bostezo", total_frames)

    vectores_cierre_a.append(v_a_c); vectores_cierre_b.append(v_b_c)
    vectores_bostezo_a.append(v_a_y); vectores_bostezo_b.append(v_b_y)

    kappa_c = cohen_kappa_score(v_a_c, v_b_c) if len(set(v_a_c)|set(v_b_c)) > 1 else np.nan
    kappa_y = cohen_kappa_score(v_a_y, v_b_y) if len(set(v_a_y)|set(v_b_y)) > 1 else np.nan
    resumen_por_video.append({"video": nombre, "kappa_cierre": kappa_c, "kappa_bostezo": kappa_y,
                               "n_eventos_A": len(eventos_a), "n_eventos_B": len(eventos_b)})

df_eventos_global = pd.DataFrame(tabla_eventos_global)
df_resumen_kappa = pd.DataFrame(resumen_por_video)

print("Eventos totales registrados:", len(df_eventos_global))
print(df_resumen_kappa)

Eventos totales registrados: 187
                video  kappa_cierre  kappa_bostezo  n_eventos_A  n_eventos_B
0    V1_video1_normal      0.908992       0.930985            5            5
1   V1_video2_pocaluz      0.925785       0.904687            5            5
2    V2_video1_normal      0.929500       0.928868            4            4
3    V2_video2_lentes      0.945844       0.937520            4            4
4    V3_video1_normal      0.816376       0.857716           13           12
5    V3_video2_lentes      0.843423       0.911114            9            9
6    V4_video1_normal      0.710130       0.880321            5            4
7   V4_video2_pocaluz      0.840820       0.927040            3            3
8    V5_video1_normal      0.847792       0.767931            4            4
9   V5_video2_pocaluz      0.825533       0.447179            4            3
10   V6_video1_normal      0.918207       0.766696            6            6
11  V6_video2_pocaluz      0.844521       0

In [27]:
vector_global_a_cierre = np.concatenate(vectores_cierre_a)
vector_global_b_cierre = np.concatenate(vectores_cierre_b)
vector_global_a_bostezo = np.concatenate(vectores_bostezo_a)
vector_global_b_bostezo = np.concatenate(vectores_bostezo_b)

kappa_global_cierre = cohen_kappa_score(vector_global_a_cierre, vector_global_b_cierre)
kappa_global_bostezo = cohen_kappa_score(vector_global_a_bostezo, vector_global_b_bostezo)

print(f"Kappa GLOBAL - cierre_ocular: {kappa_global_cierre:.3f}")
print(f"Kappa GLOBAL - bostezo: {kappa_global_bostezo:.3f}")

Kappa GLOBAL - cierre_ocular: 0.887
Kappa GLOBAL - bostezo: 0.878


In [28]:
Path("../results/ground_truth").mkdir(parents=True, exist_ok=True)
df_eventos_global.to_csv("../results/ground_truth/eventos_todos_los_videos.csv", index=False)
df_resumen_kappa.to_csv("../results/ground_truth/resumen_kappa_por_video.csv", index=False)

pd.DataFrame({
    "indicador": ["Kappa global - cierre_ocular", "Kappa global - bostezo"],
    "valor": [kappa_global_cierre, kappa_global_bostezo]
}).to_csv("../results/ground_truth/kappa_global.csv", index=False)

print("Todo guardado en results/ground_truth/")

Todo guardado en results/ground_truth/
